In [1]:
import torch
import pandas as pd
from tqdm import tqdm
from Bio.Seq import Seq
from ete3 import Tree

tqdm.pandas()

from plmr.models.frameworks.peint import (
    load_from_old_checkpoint,
    simulate_evolution_with_rejection_sampling,
)

data_dir = '/scratch/users/milind_jagota/bcr/data/heavylight/trees/'

/scratch/users/stephen.lu/uv-envs/plm/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/scratch/users/stephen.lu/uv-envs/plm/lib/python3.10/site-packages/wandb/sdk/launch/builder/build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Load data, pick family, convert to tree

In [2]:
def df_to_ete3_tree(df):
    """
    Convert a dataframe of edges (parent_name, child_name, branch_length)
    into an ete3 Tree object in 'format=3'.

    The tree is assumed to have exactly one root: a node that does not appear as any child's name.
    """

    # Create a lookup for all nodes
    # We'll store node_name -> ete3.Tree instance
    nodes = {}

    # Create all nodes, but do not connect them yet
    for parent_name, child_name, branch_length in df[['parent_name', 'child_name', 'branch_length']].itertuples(index=False):
        if parent_name not in nodes:
            nodes[parent_name] = Tree(name=parent_name)  # parent node
        if child_name not in nodes:
            nodes[child_name] = Tree(name=child_name)    # child node

    # Connect child to parent, setting the branch length (dist) on the child side
    # We'll also track who is the parent of whom in parent_of
    parent_of = {}
    for parent_name, child_name, branch_length in df[['parent_name', 'child_name', 'branch_length']].itertuples(index=False):
        parent_node = nodes[parent_name]
        child_node = nodes[child_name]
        # Set branch length on the child
        child_node.dist = branch_length
        # Attach child to parent's children
        parent_node.add_child(child_node)
        parent_of[child_name] = parent_name

    # Find the root as the node that never appears as a child
    all_nodes = set(nodes.keys())
    all_children = set(parent_of.keys())
    root_candidates = all_nodes - all_children
    if len(root_candidates) != 1:
        raise ValueError(f"Expected exactly 1 root, found: {root_candidates}")
    root_name = root_candidates.pop()

    # Return the root as an ete3 Tree
    # By construction, nodes[root_name] is now the root and has all children below it
    tree = nodes[root_name]

    # If you want to ensure 'format=3' usage downstream, you can do:
    #   newick_str = tree.write(format=3)
    #   tree = Tree(newick_str, format=3)
    # But typically, once you have 'tree' as a Tree object, you can simply
    # use `tree.write(format=3)` or pass format=3 when you want to export.
    return tree

In [3]:
heavy = pd.read_csv(data_dir + 'wyatt-10x-1p5m_paired-igh_fs-all_pcp_2024-11-21.csv.gz', compression='gzip', index_col=0)
kappa = pd.read_csv(data_dir + 'wyatt-10x-1p5m_paired-igk_fs-all_pcp_2024-11-21.csv.gz', compression='gzip', index_col=0)
lambd = pd.read_csv(data_dir + 'wyatt-10x-1p5m_paired-igl_fs-all_pcp_2024-11-21.csv.gz', compression='gzip', index_col=0)

heavy['family'] = heavy['sample_id'] + '_' + heavy['family'].astype(str)
kappa['family'] = kappa['sample_id'] + '_' + kappa['family'].astype(str)
lambd['family'] = lambd['sample_id'] + '_' + lambd['family'].astype(str)

heavy['edge_id'] = heavy['family'] + ';' + heavy['parent_name'] + ';' + heavy['child_name']
kappa['edge_id'] = kappa['family'] + ';' + kappa['parent_name'] + ';' + kappa['child_name']
lambd['edge_id'] = lambd['family'] + ';' + lambd['parent_name'] + ';' + lambd['child_name']

In [4]:
keep_kappa = (kappa.edge_id.isin(heavy.edge_id)) & ~(kappa.edge_id.isin(lambd.edge_id))
keep_lambd = (lambd.edge_id.isin(heavy.edge_id)) & ~(lambd.edge_id.isin(kappa.edge_id))
kappa = kappa[keep_kappa]
lambd = lambd[keep_lambd]
keep_heavy = (heavy.edge_id.isin(kappa.edge_id)) | (heavy.edge_id.isin(lambd.edge_id))
heavy = heavy[keep_heavy]

In [5]:
merge_cols = ['sample_id', 'family', 'parent_name', 'child_name', 'edge_id']
keep_cols = ['parent', 'child', 'branch_length', 'depth', 'distance', 'v_gene', 'cdr1_codon_start', 
             'cdr1_codon_end', 'cdr2_codon_start', 'cdr2_codon_end', 'cdr3_codon_start', 'cdr3_codon_end',
             'parent_is_naive', 'child_is_leaf']
keep_cols = merge_cols + keep_cols

In [6]:
heavy_kappa = pd.merge(heavy[keep_cols], kappa[keep_cols], on=merge_cols, how='inner', suffixes=('_heavy', '_light'))
heavy_lambd = pd.merge(heavy[keep_cols], lambd[keep_cols], on=merge_cols, how='inner', suffixes=('_heavy', '_light'))
full_df = pd.concat([heavy_kappa, heavy_lambd], axis=0)

In [7]:
family = 'd4_203694-igk-203694'
edges = full_df[full_df.family == family]

edges['parent_heavy'] = edges['parent_heavy'].progress_apply(lambda x: str(Seq(x).translate()))
edges['parent_light'] = edges['parent_light'].progress_apply(lambda x: str(Seq(x).translate()))
edges['child_heavy'] = edges['child_heavy'].progress_apply(lambda x: str(Seq(x).translate()))
edges['child_light'] = edges['child_light'].progress_apply(lambda x: str(Seq(x).translate()))

edges = edges.rename(columns={'branch_length_heavy':'branch_length'})
edges['branch_length'] /= edges['branch_length'].mean()

100%|████████████████████████████████████████████████████████████████████████████████| 291/291 [00:00<00:00, 22097.27it/s]
/tmp/ipykernel_1952803/2731458363.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  edges['parent_heavy'] = edges['parent_heavy'].progress_apply(lambda x: str(Seq(x).translate()))
100%|████████████████████████████████████████████████████████████████████████████████| 291/291 [00:00<00:00, 25613.13it/s]
/tmp/ipykernel_1952803/2731458363.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [8]:
tree = df_to_ete3_tree(edges[['parent_name', 'child_name', 'branch_length']])

root_data = edges[edges.parent_name == 'naive'].iloc[0][['parent_heavy', 'parent_light']]
root_seq = root_data.parent_heavy + ('G' * 10) + root_data.parent_light

leaf_data = edges[edges.child_name.str.contains('contig')][['child_name', 'child_heavy', 'child_light']]
leaf_data['true'] = leaf_data.child_heavy + ('G' * 10) + leaf_data.child_light

In [9]:
tree.write()

'(((((((((((((((((AAACGGGTCATTCACT-1-1279069_contig_h:0.267324,CAGATCAAGGACACCA-1-1287171_contig_h:0.112265)1:1.77428,GCCTCTAAGACTGGGT-1-1287174_contig_h:1.58973)1:0.40775,(CGGACTGGTGAGGCTA-1-1287171_contig_h:1.7357,CTCGTCAAGTACGTTC-1-1279070_contig_h:1.64579)1:0.700829)1:0.105015,(ACTGTCCAGTATCTCG-1-1287172_contig_h:0.337478,CATCGGGGTCAACATC-1-1287160_contig_h:0.254484)1:1.52896)1:0.168921,((ACGAGGACAATCGGTT-1-1287173_contig_h:0.453492,AGAGCGAGTAATCACC-1-1287167_contig_h:0.273394)1:1.16225,GGGTCTGAGAGTGAGA-1-1287163_contig_h:0.675172)1:1.05803)1:0.182558,(((AGCAGCCAGTTAACGA-1-1287175_contig_h:3.85783,GCTGGGTCATTGGGCC-1-1287168_contig_h:2.23527)1:0.167273,CACAAACAGCAATATG-1-1279070_contig_h:4.07592)1:0.213428,((((((((ATGAGGGCATTAGCCA-1-1279070_contig_h:4.49813e-05,CCAATCCGTTGCGCAC-1-1279070_contig_h:4.49813e-05)1:0.192773,((CAGCTGGAGTGGGCTA-1-1287162_contig_h:0.383574,GAATAAGGTTGAACTC-1-1287160_contig_h:0.371086)1:0.0187813,(CGAACATGTTTAGGAA-1-1287163_contig_h:0.0630739,GCAGCCACATGAAGT

In [10]:
print(tree.get_ascii())


                                                                                            /-AAACGGGTCATTCACT-1-1279069_contig_h
                                                                                      /Node16
                                                                                /Node15     \-CAGATCAAGGACACCA-1-1287171_contig_h
                                                                               |     |
                                                                          /Node14     \-GCCTCTAAGACTGGGT-1-1287174_contig_h
                                                                         |     |
                                                                         |     |      /-CGGACTGGTGAGGCTA-1-1287171_contig_h
                                                                    /Node13     \Node17
                                                                   |     |            \-CTCGTCAAGTACGTTC-1-1279070_contig_h
                 

## Load model, run sampling

In [9]:
model_checkpoint_path = '/scratch/users/milind_jagota/bcr/models/peint/joint/epoch=28-step=12000.ckpt'

device = torch.device("cuda")

model, vocab = load_model(
    model_checkpoint_path = model_checkpoint_path,
    device = device,
    model_type = 'Cached_Transformer'
)

/accounts/projects/yss/milind_jagota/protein-evolution/protevo/simulation/_simulate_on_tree.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(model_checkp

In [ ]:
simulation_args = {
    'model': model,
    'vocab': vocab,
    'root_sequence': root_seq,
    'tree': tree,
    'device': device,
    'max_decode_steps': len(root_seq) + 1,
    'max_batch_size': 128,
    'n_sequences': 4,
    'p_threshold': 1.,
    'length_criterion': lambda x: x == len(root_seq),
    'likelihood_fn': None,
    'max_retries': 3,
    'seed': 42,
}

In [11]:
for i in tqdm(range(50)):
    simulation_args['seed'] = i
    output = simulate_evolution_with_rejection_sampling(**simulation_args)
    sim_leaves = [output[node] for node in leaf_data.child_name]
    leaf_data['sim_{0}'.format(i)] = sim_leaves

100%|██████████| 50/50 [32:34<00:00, 39.10s/it]


In [12]:
leaf_data.to_csv('real_and_sim_leaves.csv', index=False)

In [ ]:
leaf_data